In [1]:
import os
import sys
import importlib
from pathlib import Path
import loaders
sys.path.insert(0, str(Path.home() / "project/crc-pm/code"))
import src
import anndata as ad
import pandas as pd

##### current practise for GTF mapping:
Li_2025_NatCa: GRCh38.111 \
Berlin_2023_JCliInv:GRCh38.98 \
Lenos_2022_NatCom: GRCh38.98 \
Pelka_2021_Cell: GRCh38.84 \
Lee_2020_NatGen: GRCh38.84 

In [2]:
STUDIES = []
for name in [study.strip('.py') for study in os.listdir('./loaders')]:
    if name == '__pycache__':
        continue
    importlib.import_module(f".{name}", package = 'loaders')
    STUDIES.append(name)

In [3]:
adata_dict = {}
for study in STUDIES:
    # study = 'Li_2025_NatCa'
    module = getattr(loaders, study)
    adata_dict[study] = module.load()

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
from collections import Counter
from math import floor

# calculate thres
thres=0.75
min_studies = floor(len(STUDIES) * thres)

gene_counter = Counter()
for adata in adata_dict.values():
    if not adata.var_names.is_unique:
        raise ValueError("adata.var_names must be unique.")
    gene_counter.update(adata.var_names)
keep_genes = {gene for gene, count in gene_counter.items() if count >= min_studies}

In [5]:
# concatenate
concat_list = [adata[:, adata.var_names.isin(keep_genes)].copy() for adata in adata_dict.values()]
def merge_var(x):
    if isinstance(x[0], pd.DataFrame):
        return pd.concat(x).groupby(level=0, sort=False).first()
    return {}
concat_adata = ad.concat(
    concat_list,
    axis="obs",
    join="outer",
    merge=merge_var,
    fill_value=0,
)
print(concat_adata.var.isna().sum())

gene_name       0
gene_id         0
gene_biotype    0
dtype: int64


In [6]:
EXPORTDIR = Path("../../result/curated/merged.h5ad")
concat_adata.write_h5ad(EXPORTDIR)